> **Versión t70:** copia de `03_gradcam_moondream2.ipynb` con figuras de 70 px. Usa `conjunto_t70.zip` y guarda en `salidas/03_gradcam_t70/`; la versión de 35 px no cambia.

# 03 · Grad-CAM de moondream2 (depende de la pregunta)

Para cada una de las 20 preguntas de la muestra (imagen original `c0`):
1. Se hace la misma pregunta que en el notebook 01 (mismo sufijo, temperature 0), en una sola pasada **con gradientes**.
2. Se toma el **logit del primer token de la respuesta** (el número que el modelo elige) y se calcula su gradiente hacia atrás.
3. Se arman tres mapas de 27 × 27:
   - **Grad-CAM LLM**: sobre los 729 tokens de imagen que recibe el LLM (la vista global y el zoom ya fusionados). ReLU(Σ_d α_d · E_d), con α = gradiente promediado sobre los tokens. Es el mapa principal.
   - **Grad × entrada LLM**: Σ_d g_d · E_d por token, con signo.
   - **Grad-CAM encoder**: lo mismo, sobre la salida del encoder visual en la vista global (antes de fusionarla con el zoom).

Controles: imagen gris y de ruido con la pregunta "How many circles are there in the image?".

Una celda verifica que esta pasada da el **mismo número** que `model.query()`.

**Cómo correrlo:** igual que los anteriores. T4 GPU → *Ejecutar todas* (si se reinicia, otra vez). ~5–10 min. Los resultados quedan en `memoria_explicabilidad/salidas/03_gradcam_t70/`.

In [1]:
import importlib.metadata, os
VERSION = "4.46.3"
if importlib.metadata.version("transformers") != VERSION:
    os.system(f"pip install -q transformers=={VERSION}")
    print("Instalado. La sesión se reinicia: vuelve a 'Ejecutar todas'.")
    os.kill(os.getpid(), 9)
print("transformers", VERSION, "OK")

transformers 4.46.3 OK


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob, zipfile
BASE = "/content/drive/MyDrive/memoria_explicabilidad"
SALIDA = f"{BASE}/salidas/03_gradcam_t70"
os.makedirs(f"{SALIDA}/png", exist_ok=True)
encontrados = sorted(glob.glob(f"{BASE}/**/conjunto_t70*.zip", recursive=True))
assert encontrados, "No encontré conjunto_t70.zip dentro de memoria_explicabilidad/"
with zipfile.ZipFile(encontrados[0]) as z:
    z.extractall("/content")
CONJUNTO = "/content/conjunto_t70"

Mounted at /content/drive


In [3]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "vikhyatk/moondream2"
MODEL_REVISION = "2025-06-21"
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
    torch_dtype=torch.bfloat16, device_map={"": "cuda"})
model.requires_grad_(False)   # solo interesa el gradiente respecto de la imagen, no de los pesos
md = model.model
print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

hf_moondream.py: 0.00B [00:00, ?B/s]

moondream.py: 0.00B [00:00, ?B/s]

image_crops.py: 0.00B [00:00, ?B/s]

text.py: 0.00B [00:00, ?B/s]

... (7 outputs left out)

model.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

GPU: Tesla T4


In [4]:
import sys, re
def buscar(sufijo, atributo):
    for nombre, mod in list(sys.modules.items()):
        if mod is not None and "moondream" in nombre.lower() and nombre.endswith(sufijo) and hasattr(mod, atributo):
            return mod
    raise RuntimeError(f"No encontré el módulo {sufijo} con {atributo}")
text_mod = buscar(".text", "text_decoder")
TCFG = md.config.text
SUFIJO = " Respond with only the integer number."

def extraer_entero(texto):
    limpio = re.sub(r"(?m)^\s*\d+[.)]\s+", "", str(texto))
    m = re.search(r"-?\d+", limpio)
    return int(m.group()) if m else None

def gradcam(imagen, pregunta):
    """Una pasada con gradientes: imagen + pregunta -> logit del primer token de respuesta -> mapas."""
    captura = {}
    vis_enc_original = md._vis_enc
    def vis_enc_con_captura(x):
        salida = vis_enc_original(x).detach().requires_grad_(True)   # salida del encoder (todos los recortes)
        captura["enc"] = salida
        return salida
    md._vis_enc = vis_enc_con_captura
    try:
        with torch.enable_grad():
            emb = md._run_vision_encoder(imagen)                        # [729, 2048]: tokens de imagen para el LLM
            emb.retain_grad()
            tok = md.config.tokenizer
            ids = tok.templates["query"]["prefix"] + md.tokenizer.encode(pregunta + SUFIJO).ids + tok.templates["query"]["suffix"]
            bos = text_mod.text_encoder(torch.tensor([[tok.bos_id]], device=md.device), md.text)
            texto = text_mod.text_encoder(torch.tensor([ids], device=md.device), md.text)
            x = torch.cat([bos, emb[None].to(bos.dtype), texto], dim=1)
            L = x.shape[1]
            mascara = md.attn_mask[:, :, :L, :L]                        # imagen bidireccional, texto causal
            pos = torch.arange(L, dtype=torch.long)
            for block in md.text.blocks:                                # = text_decoder(), sin caché
                l_in = text_mod.layer_norm(x, block.ln)
                x = x + text_mod.attn(l_in, block.attn, freqs_cis=md.text.freqs_cis, kv_cache=None,
                                      attn_mask=mascara, n_heads=TCFG.n_heads, n_kv_heads=TCFG.n_kv_heads,
                                      position_ids=pos, lora=None) + text_mod.mlp(l_in, block.mlp)
            logits = text_mod.lm_head(x, md.text)[0].float()
            pred = int(logits.argmax())
            logits[pred].backward()
    finally:
        md._vis_enc = vis_enc_original
    E, G = emb.detach().float(), emb.grad.float()
    A, GA = captura["enc"][0].detach().float(), captura["enc"].grad[0].float()
    return dict(
        gradcam_llm=torch.relu((E * G.mean(0)).sum(-1)).cpu().numpy(),
        gxi_llm=(E * G).sum(-1).cpu().numpy(),
        gradcam_enc=torch.relu((A * GA.mean(0)).sum(-1)).cpu().numpy(),
        token=md.tokenizer.decode([pred]), logit=float(logits[pred]),
        prob=float(logits.softmax(-1)[pred]))

In [6]:
# Verificación: la pasada con gradientes elige el mismo número que model.query()
import csv, json, numpy as np
from PIL import Image
MUESTRA = {'q040': 'acierto', 'q022': 'acierto', 'q018': 'acierto', 'q007': 'acierto', 'q108': 'acierto', 'q016': 'acierto', 'q074': 'acierto', 'q054': 'cercano', 'q010': 'cercano', 'q085': 'cercano', 'q017': 'cercano', 'q011': 'cercano', 'q093': 'cercano', 'q089': 'cercano', 'q113': 'lejano', 'q078': 'lejano', 'q056': 'lejano', 'q029': 'lejano', 'q028': 'lejano', 'q079': 'lejano'}  # t70: PEGAR aquí el diccionario de 20 preguntas que imprime el paso B6
assert MUESTRA, "Falta pegar MUESTRA: ver el paso B6 (muestra_mapas.csv de 01_conducta_t70)"
preguntas = {p["qid"]: p for p in csv.DictReader(open(f"{CONJUNTO}/preguntas.csv"))}

coinciden = 0
for q in list(MUESTRA)[:5]:
    img = Image.open(f"{CONJUNTO}/imagenes/{q}_c0.png").convert("RGB")
    r = gradcam(img, preguntas[q]["pregunta"])
    resp = model.query(model.encode_image(img), preguntas[q]["pregunta"] + SUFIJO, settings={"temperature": 0, "max_tokens": 16})["answer"]
    ok = extraer_entero(r["token"]) == extraer_entero(resp)
    coinciden += ok
    print(q, "| gradiente:", repr(r["token"]), "| query:", repr(resp), "| OK" if ok else "| DISTINTO")
assert coinciden == 5, "La pasada con gradientes no reproduce a query(): no sigas, avísame."

/tmp/ipykernel_917/3848763672.py:53: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  token=md.tokenizer.decode([pred]), logit=float(logits[pred]),


q040 | gradiente: '2' | query: '2' | OK
q022 | gradiente: '3' | query: '3' | OK
q018 | gradiente: '3' | query: '3' | OK
q007 | gradiente: '2' | query: '2' | OK
q108 | gradiente: '8' | query: '8' | OK


In [7]:
import matplotlib.pyplot as plt
LADO = 27
imagenes = {q: (Image.open(f"{CONJUNTO}/imagenes/{q}_c0.png").convert("RGB"), preguntas[q]["pregunta"]) for q in MUESTRA}
CONTROL = "How many circles are there in the image?"
imagenes["gris"] = (Image.new("RGB", (1024, 1024), (200, 200, 200)), CONTROL)
imagenes["ruido"] = (Image.fromarray(np.random.default_rng(0).integers(0, 256, (1024, 1024, 3), dtype=np.uint8)), CONTROL)

mapas, filas = {}, []
for ident, (img, pregunta) in imagenes.items():
    r = gradcam(img, pregunta)
    for k in ["gradcam_llm", "gxi_llm", "gradcam_enc"]:
        mapas[f"{ident}_{k}"] = r[k]
    top = np.argsort(r["gradcam_llm"])[::-1][:5]
    filas.append(dict(id=ident, grupo=MUESTRA.get(ident, "control"), pregunta=pregunta,
                      token=r["token"], prediccion=extraer_entero(r["token"]), logit=round(r["logit"], 3),
                      prob=round(r["prob"], 4), top5_gradcam_llm=" ".join(map(str, top))))
    fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
    ax[0].imshow(img); ax[0].set_title(f"{ident} ({MUESTRA.get(ident, 'control')}) → {r['token'].strip()}")
    for eje, k, t in [(ax[1], "gradcam_llm", "Grad-CAM (tokens del LLM)"), (ax[2], "gradcam_enc", "Grad-CAM (encoder, vista global)")]:
        eje.imshow(img); eje.imshow(r[k].reshape(LADO, LADO), cmap="jet", alpha=0.55,
                                     extent=(0, img.width, img.height, 0), interpolation="nearest")
        eje.set_title(t)
    for eje in ax: eje.axis("off")
    fig.suptitle(pregunta, fontsize=9)
    plt.tight_layout(); plt.savefig(f"{SALIDA}/png/{ident}.png", dpi=110); plt.close(fig)
    print("OK", ident, "| respuesta:", repr(r["token"]), "| top-5 Grad-CAM:", top)

np.savez_compressed(f"{SALIDA}/mapas_gradcam.npz", **mapas)
with open(f"{SALIDA}/resumen.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0])); w.writeheader(); w.writerows(filas)

OK q040 | respuesta: '2' | top-5 Grad-CAM: [137 119 164 138 172]
OK q022 | respuesta: '3' | top-5 Grad-CAM: [  7 139 174 406 137]
OK q018 | respuesta: '3' | top-5 Grad-CAM: [ 49 681 411 680 410]
OK q007 | respuesta: '2' | top-5 Grad-CAM: [516 301 697   7 723]
OK q108 | respuesta: '8' | top-5 Grad-CAM: [  9 585  89 624 714]
OK q016 | respuesta: '2' | top-5 Grad-CAM: [549 403 490 430 548]
OK q074 | respuesta: '3' | top-5 Grad-CAM: [292 126  24 291 220]
OK q054 | respuesta: '3' | top-5 Grad-CAM: [410 722 514 411 695]
OK q010 | respuesta: '2' | top-5 Grad-CAM: [723 695   7 651 673]
OK q085 | respuesta: '6' | top-5 Grad-CAM: [566 280 271 284 282]
OK q017 | respuesta: '2' | top-5 Grad-CAM: [540 541 515 513 543]
OK q011 | respuesta: '3' | top-5 Grad-CAM: [687 714 268 184 556]
OK q093 | respuesta: '6' | top-5 Grad-CAM: [707 714 595 409 411]
OK q089 | respuesta: '6' | top-5 Grad-CAM: [136 551 722 720 163]
OK q113 | respuesta: '6' | top-5 Grad-CAM: [722 163 312 720 136]
OK q078 | respuesta: '4' 

In [8]:
import transformers, platform
info = dict(modelo=MODEL_ID, revision=MODEL_REVISION, transformers=transformers.__version__, torch=torch.__version__,
            gpu=torch.cuda.get_device_name(0), python=platform.python_version(), sufijo=SUFIJO,
            objetivo_gradiente="logit del primer token de respuesta (argmax)", verificacion_coinciden_de_5=coinciden,
            vram_pico_mb=round(torch.cuda.max_memory_allocated() / 1024**2))
json.dump(info, open(f"{SALIDA}/info_corrida.json", "w"), indent=2)
g = set(filas[-2]["top5_gradcam_llm"].split())
print("Top-5 de Grad-CAM compartidos con la imagen gris (de 5), por imagen:",
      [len(g & set(f["top5_gradcam_llm"].split())) for f in filas[:-2]])
print("\nLISTO. Archivos en", SALIDA, ":", sorted(os.listdir(SALIDA)))

Top-5 de Grad-CAM compartidos con la imagen gris (de 5), por imagen: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]

LISTO. Archivos en /content/drive/MyDrive/memoria_explicabilidad/salidas/03_gradcam_t70 : ['info_corrida.json', 'mapas_gradcam.npz', 'png', 'resumen.csv']
